# Radionica 5: LLM modeli i prompting

**GirlTHing AI Lab — Blok 2**

> Ćelije označene sa ✏️ **ZADATAK** popunjavate sami. Ostale ćelije samo pokrenite i pročitajte rezultat.

## Ciljevi radionice

Nakon ove radionice znaš:

1. pozvati OpenAI API iz Pythona i pročitati odgovor,
2. objasniti šta su tokeni, kontekstni prozor i koliko košta jedan poziv,
3. zašto model ponekad "izmišlja" (halucinira) i kako to smanjiti,
4. pisati dobre promptove: instrukcije, few-shot primjeri, struktura, JSON izlaz,
5. napraviti mali chatbot koji pamti razgovor.

Kroz cijeli Blok 2 gradimo isti primjer: **Studentski asistent**, chatbot koji studenticama pomaže oko CV-a, praksi, stipendija i učenja.

## 0. Priprema

U Colabu: lijevo klikni ikonu 🔑 (Secrets), dodaj `OPENAI_API_KEY` i uključi *Notebook access*.

⚠️ API ključ nikad ne pišemo direktno u kod. Ko ima ključ, troši naš budžet.

In [ ]:
# Instalacija paketa (u Colabu je potrebno samo jednom po sesiji)
%pip install -q openai numpy pydantic python-dotenv pypdf

import os
from openai import OpenAI

# API ključ: u Colabu iz "Secrets" (ikona ključa lijevo), lokalno iz .env datoteke
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

client = OpenAI()
MODEL = "gpt-6-luna"     # jeftin i brz model, dovoljan za sve vježbe
print("Spremno ✅")

## 1. Prvi poziv modela

Koristimo **Responses API**: šaljemo `model` i `input`, a dobijamo objekat čiji je tekst u `output_text`.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Objasni u dvije rečenice šta je mašinsko učenje.",
)
print(response.output_text)

## 2. Tokeni i cijena

Model ne čita slova ni riječi, nego **tokene** (dijelove riječi). Na engleskom je jedan token u prosjeku oko 4 znaka; na bosanskom riječi se češće dijele na više tokena.

Plaćamo posebno **ulazne** tokene (sve što pošaljemo) i **izlazne** tokene (sve što model napiše).
Cijena za `gpt-6-luna`: **$0.10** po milion ulaznih i **$0.50** po milion izlaznih tokena.

**Kontekstni prozor** je maksimalan broj tokena koje model može "vidjeti" odjednom (pitanje + historija + dokumenti + odgovor).

In [ ]:
u = response.usage
print("Ulazni tokeni:", u.input_tokens)
print("Izlazni tokeni:", u.output_tokens)

PRICE_IN, PRICE_OUT = 0.10, 0.50          # $ po milionu tokena
cost = u.input_tokens / 1e6 * PRICE_IN + u.output_tokens / 1e6 * PRICE_OUT
print(f"Cijena ovog poziva: ${cost:.6f}")

### ✏️ ZADATAK 1
Recimo da 15 učesnica pošalje po 200 ovakvih pitanja. Koliko bi to koštalo? A koliko bi koštalo s modelom koji je 20 puta skuplji?

In [ ]:
broj_poziva = ...
ukupno = ...
print(f"Ukupno: ${ukupno:.4f}")

## 3. Instrukcije (system prompt)

Parametar `instructions` govori modelu **ko je i kako se ponaša**. Instrukcije imaju prednost nad porukom korisnika.

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions="Ti si strpljiva mentorica za programiranje. Odgovaraš kratko, na bosanskom, i uvijek završiš jednim pitanjem za provjeru razumijevanja.",
    input="Šta je rekurzija?",
)
print(response.output_text)

### ✏️ ZADATAK 2
Napiši instrukcije za **Studentskog asistenta**. Neka chatbot:
- predstavi se imenom,
- odgovara samo na teme karijere, studija i stipendija (za ostalo ljubazno odbije),
- odgovara u najviše 5 rečenica.

Testiraj s jednim pitanjem iz teme i jednim izvan teme (npr. recept za kolače).

In [ ]:
MOJE_INSTRUKCIJE = """
...
"""

for pitanje in ["Kako da napišem CV bez radnog iskustva?", "Daj mi recept za čokoladnu tortu."]:
    r = client.responses.create(model=MODEL, instructions=MOJE_INSTRUKCIJE, input=pitanje)
    print("❓", pitanje, "\n💬", r.output_text, "\n")

## 4. Isti prompt, različiti odgovori

LLM bira sljedeći token **vjerovatnosno**, pa isti prompt može dati različite odgovore.
Kod nekih modela to se podešava parametrom `temperature` (0 = predvidljivije, veće vrijednosti = kreativnije). Noviji *reasoning* modeli ne podržavaju uvijek `temperature`, pa kod njih ponašanje kontrolišemo samim promptom.

In [ ]:
for i in range(3):
    r = client.responses.create(model=MODEL, input="Smisli kratak, originalan naziv za chatbot koji pomaže studenticama. Samo naziv.")
    print(i + 1, r.output_text)

try:
    r = client.responses.create(model=MODEL, input="Smisli naziv za chatbot. Samo naziv.", temperature=0)
    print("temperature=0:", r.output_text)
except Exception as e:
    print("Ovaj model ne podržava temperature:", type(e).__name__)

## 5. Halucinacije

Model ne "zna" činjenice kao baza podataka. On generiše **uvjerljiv** tekst, pa ponekad izmisli odgovor koji zvuči tačno.

In [ ]:
pitanje = "Koja studentica je osvojila nagradu GirlTHing AI Challenge 2026. i šta je bio njen projekat?"

r1 = client.responses.create(model=MODEL, input=pitanje)
print("BEZ upozorenja:\n", r1.output_text, "\n")

r2 = client.responses.create(
    model=MODEL,
    instructions="Ako nisi siguran/na u činjenicu, reci 'Ne znam' umjesto da pogađaš.",
    input=pitanje,
)
print("S upozorenjem:\n", r2.output_text)

💡 Instrukcija pomaže, ali **ne rješava** problem. Pravo rješenje je da modelu damo tačne informacije u promptu. To je **RAG**, tema sljedeće radionice.

## 6. Prompt engineering: zero-shot i few-shot

Zadatak: chatbot treba svrstati pitanje studentice u jednu kategoriju: `CV`, `PRAKSA`, `STIPENDIJA`, `UCENJE`, `OSTALO`.

- **Zero-shot**: samo opišemo zadatak.
- **Few-shot**: dodamo nekoliko primjera ulaza i izlaza.

In [ ]:
PITANJA = [
    "Da li da stavim sliku na CV?",
    "Kad počinju prijave za ljetne prakse u IT firmama?",
    "Kako da ne zaboravim gradivo do ispita?",
    "Ima li općina neku pomoć za studente informatike?",
    "Koji laptop da kupim?",
]

zero_shot = "Svrstaj pitanje u jednu kategoriju: CV, PRAKSA, STIPENDIJA, UCENJE ili OSTALO."

for p in PITANJA:
    r = client.responses.create(model=MODEL, instructions=zero_shot, input=p)
    print(f"{r.output_text!r:40} ← {p}")

Primijeti: odgovori nisu uvijek u istom obliku (nekad cijela rečenica, nekad samo riječ). To otežava korištenje u kodu.

### ✏️ ZADATAK 3
Napiši **few-shot** instrukcije s 3–5 primjera tako da model vraća **samo** naziv kategorije, velikim slovima.

In [ ]:
few_shot = """
...
"""

for p in PITANJA:
    r = client.responses.create(model=MODEL, instructions=few_shot, input=p)
    print(f"{r.output_text!r:15} ← {p}")

## 7. Struktura dobrog prompta

Dobar prompt najčešće ima ove dijelove:

| Dio | Primjer |
| --- | --- |
| **Uloga** | Ti si karijerna savjetnica za studente IT-a. |
| **Kontekst** | Korisnica je studentica 3. godine bez radnog iskustva. |
| **Zadatak** | Napiši opis projekta za CV. |
| **Format** | Jedna rečenica, najviše 30 riječi, počni glagolom. |
| **Ograničenja** | Ne izmišljaj tehnologije koje nisu navedene. |
| **Primjer** | "Razvila sam ... koristeći ..., što je ..." |

### ✏️ ZADATAK 4: Prompt battle 🏆
Svaka učesnica piše prompt koji od opisa projekta pravi **jednu rečenicu za CV**. Najbolji rezultat (jasno, konkretno, s rezultatom) pobjeđuje!

Opis projekta: *"radila sam neki program u pythonu sa podacima o stanovima u tuzli, gledala sam koje cijene i napravila model koji predviđa cijenu, bilo je ok tačno"*

In [ ]:
OPIS = "radila sam neki program u pythonu sa podacima o stanovima u tuzli, gledala sam koje cijene i napravila model koji predviđa cijenu, bilo je ok tačno"

MOJ_PROMPT = """
...
"""

r = client.responses.create(model=MODEL, instructions=MOJ_PROMPT, input=OPIS)
print(r.output_text)

## 8. Strukturiran izlaz (JSON)

Kad odgovor modela koristimo u kodu, želimo **tačno određen format**. Uz `client.responses.parse` i Pydantic klasu model garantovano vraća podatke u zadanoj strukturi.

In [ ]:
from pydantic import BaseModel
from typing import Literal

class AnalizaPitanja(BaseModel):
    kategorija: Literal["CV", "PRAKSA", "STIPENDIJA", "UCENJE", "OSTALO"]
    hitno: bool
    kratak_odgovor: str

r = client.responses.parse(
    model=MODEL,
    instructions="Analiziraj pitanje studentice. 'hitno' je True ako se spominje rok u narednih 7 dana.",
    input="Sutra mi je rok za prijavu na praksu, a nemam propratno pismo. Šta da napišem?",
    text_format=AnalizaPitanja,
)
analiza = r.output_parsed
print(analiza)
print("Kategorija:", analiza.kategorija, "| Hitno:", analiza.hitno)

### ✏️ ZADATAK 5
Napravi klasu `OglasZaPosao` s poljima `pozicija`, `kompanija`, `grad`, `tehnologije` (lista) i `junior` (bool). Iz teksta oglasa izvuci podatke.

In [ ]:
OGLAS = """Tražimo Junior Python developericu/developera za naš tim u Tuzli.
Potrebno: Python, osnove SQL-a i Git. Poželjno: iskustvo s FastAPI.
Kompanija: DataNova d.o.o. Prijave do 30.11."""

class OglasZaPosao(BaseModel):
    ...

# r = client.responses.parse(...)
# print(r.output_parsed)

## 9. Mini projekat: chatbot koji pamti razgovor

Model **ne pamti** prethodne pozive. Svaki poziv je nezavisan. Da bi chatbot "pamtio", mi čuvamo historiju poruka i šaljemo je svaki put.

In [ ]:
INSTRUKCIJE = "Ti si Studentski asistent. Odgovaraš kratko i prijateljski, na bosanskom."

history = []   # lista poruka: {"role": "user" ili "assistant", "content": "..."}

def chat(poruka: str) -> str:
    history.append({"role": "user", "content": poruka})
    r = client.responses.create(model=MODEL, instructions=INSTRUKCIJE, input=history)
    history.append({"role": "assistant", "content": r.output_text})
    return r.output_text

print(chat("Zdravo! Ja sam Amra, studiram računarstvo na 2. godini."))
print("---")
print(chat("Kako se zovem i na kojoj sam godini?"))

### ✏️ ZADATAK 6
Historija stalno raste, pa svaki poziv postaje skuplji. Izmijeni funkciju tako da se modelu šalje **samo zadnjih 6 poruka**. Zatim provjeri broj ulaznih tokena prije i poslije izmjene.

*Bonus:* Responses API može sam čuvati historiju preko parametra `previous_response_id`. Pronađi u dokumentaciji kako.

In [ ]:
MAX_PORUKA = 6
history = []

def chat(poruka: str) -> str:
    history.append({"role": "user", "content": poruka})
    r = client.responses.create(model=MODEL, instructions=INSTRUKCIJE, input=history)  # TODO: šalji samo zadnjih MAX_PORUKA
    history.append({"role": "assistant", "content": r.output_text})
    print(f"   [ulaznih tokena: {r.usage.input_tokens}]")
    return r.output_text

for p in ["Zdravo, ja sam Amra.", "Šta je praksa?", "Koliko traje?", "Da li je plaćena?", "Kako se zovem?"]:
    print("👩", p, "\n🤖", chat(p))

### Interaktivni chat (opcionalno)
Pokreni ćeliju ispod i razgovaraj sa svojim chatbotom. Upiši `kraj` za izlaz.

In [ ]:
# Odkomentariši za interaktivni razgovor:
# while (poruka := input("Ti: ")) != "kraj":
#     print("Asistent:", chat(poruka))

## 📝 Domaći zadatak

1. Završi zadatke koje nisi stigla na radionici.
2. Razmisli o temi chatbota koji bi ti bio koristan (obrazovanje, karijera, podrška mladima).
3. Pronađi i sačuvaj **5–10 stvarnih dokumenata** na tu temu (PDF, tekst s web stranica, pravilnici, vodiči). Na sljedećoj radionici od njih pravimo bazu znanja.

## Sažetak

| Pojam | Ukratko |
| --- | --- |
| Token | Dio riječi; po tokenima se plaća i mjeri kontekst |
| Instrukcije | Govore modelu ko je i kako se ponaša |
| Halucinacija | Uvjerljiv, ali netačan odgovor |
| Few-shot | Primjeri u promptu koji pokazuju željeni izlaz |
| Strukturiran izlaz | Model vraća podatke u tačno zadanom obliku |
| Historija | Model ne pamti; mi šaljemo prethodne poruke |